### ЗАДАЧА: Панель складской инвентаризации по паттерну `MVC`

Команда warehouse operations разбирает кейсы по расхождениям после инвентаризации.
Для каждого кейса нужно хранить:
- `case_id` — идентификатор кейса;
- `sku` — товар;
- `warehouse` — склад;
- `expected_qty` — ожидаемый остаток по системе;
- `counted_qty` — фактически пересчитанный остаток;
- `unit_cost` — себестоимость одной единицы;
- `discrepancy_qty` — расхождение по количеству;
- `discrepancy_value` — стоимость расхождения;
- `status` — статус кейса;
- `auditor` — сотрудник, который ведет кейс;
- `recount_done` — был ли выполнен повторный пересчет;
- `decision` — финальное решение.

Нужно реализовать внутреннюю консольную панель по паттерну `MVC`, где:
- `Model` хранит кейсы и бизнес-правила;
- `View` отвечает только за отображение;
- `Controller` вызывает модель и передает результат во view.

### Статусы кейса

- `new` — кейс создан, но еще не взят в работу;
- `auditing` — аудитор начал проверку;
- `recounted` — выполнен повторный пересчет;
- `ready_for_adjustment` — кейс готов к финальной корректировке;
- `adjusted` — расхождение подтверждено и проведено;
- `escalated` — кейс передан старшему специалисту.

### Формулы

Для каждого кейса нужно правильно считать поля:
- `discrepancy_qty = counted_qty - expected_qty`
- `discrepancy_value = abs(discrepancy_qty) * unit_cost`

Также нужна агрегированная сводка:
- количество кейсов по статусам;
- `total_loss_value` — сумма стоимостей кейсов, где `discrepancy_qty < 0`;
- `total_excess_value` — сумма стоимостей кейсов, где `discrepancy_qty > 0`.

### Бизнес-правила

- нельзя создать кейс с уже существующим `case_id`;
- нельзя назначить аудитора несуществующему кейсу;
- финальные кейсы (`adjusted`, `escalated`) нельзя менять дальше;
- начать аудит можно только из `new` и только если назначен `auditor`;
- повторный пересчет можно сделать только из `auditing`;
- при пересчете нужно обновить `counted_qty`, `discrepancy_qty`, `discrepancy_value`, `recount_done` и статус;
- перевод в `ready_for_adjustment` возможен только из `auditing` или `recounted`;
- перевод в `ready_for_adjustment` нельзя делать, если `discrepancy_qty == 0`;
- завершить кейс как `adjusted` можно только из `ready_for_adjustment`;
- эскалировать кейс можно только из `auditing`, `recounted` или `ready_for_adjustment`.

In [2]:
from dataclasses import dataclass


initial_cases = [
    ("WA-100", "SKU-001", "MSK-A", 120, 108, 15.0),
    ("WA-101", "SKU-002", "SPB-2", 80, 92, 22.5),
]

actions = [
    ("show",),
    ("audit", "WA-100"),
    ("assign", "WA-100", "Olga"),
    ("audit", "WA-100"),
    ("ready", "WA-100"),
    ("recount", "WA-100", 110),
    ("ready", "WA-100"),
    ("adjust", "WA-100", "inventory_writeoff"),
    ("create", "WA-102", "SKU-003", "EKB-1", 40, 40, 50.0),
    ("assign", "WA-102", "Max"),
    ("audit", "WA-102"),
    ("ready", "WA-102"),
    ("escalate", "WA-102"),
    ("show",),
]


@dataclass
class WarehouseAuditCase:
    case_id: str
    sku: str
    warehouse: str
    expected_qty: int
    counted_qty: int
    unit_cost: float
    discrepancy_qty: int
    discrepancy_value: float
    status: str = "new"
    auditor: str | None = None
    recount_done: bool = False
    decision: str | None = None


class WarehouseAuditModel:
    def __init__(self):
        self.cases = {}
        self.done_statuses = {'adjusted','escalated'}

    def _calculate_discrepancy_qty(self, expected_qty: int, counted_qty: int) -> int:
        return counted_qty - expected_qty

    def _calculate_discrepancy_value(self, discrepancy_qty: int, unit_cost: float) -> float:
        return abs(discrepancy_qty) * unit_cost

    def add_case(self, case_id: str, sku: str, warehouse: str, expected_qty: int, counted_qty: int, unit_cost: float) -> None:
        if case_id in self.cases:
            raise ValueError('already exists')
        
        discrepancy_qty = self._calculate_discrepancy_qty(expected_qty, counted_qty)
        discrepancy_value = self._calculate_discrepancy_value(discrepancy_qty, unit_cost)
        self.cases[case_id] = WarehouseAuditCase(
            case_id,
            sku,
            warehouse,
            expected_qty,
            counted_qty,
            unit_cost,
            discrepancy_qty,
            discrepancy_value,
        )

    def assign_auditor(self, case_id: str, auditor: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        
        self.cases[case_id].auditor = auditor

    def start_audit(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status  in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != 'new':
            raise ValueError('not new')
        if self.cases[case_id].auditor is None:
            raise ValueError('no auditor')

        self.cases[case_id].status = "auditing"

    def recount_case(self, case_id: str, new_counted_qty: int) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status  in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != 'auditing':
            raise ValueError('not auditing')

        case = self.cases[case_id]
        case.counted_qty = new_counted_qty
        case.discrepancy_qty = self._calculate_discrepancy_qty(case.expected_qty, new_counted_qty)
        case.discrepancy_value = self._calculate_discrepancy_value(case.discrepancy_qty, case.unit_cost)
        case.recount_done = True
        case.status = "recounted"

    def mark_ready_for_adjustment(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        
        if self.cases[case_id].status not in  {"auditing","recounted"}:
            raise ValueError('not "auditing" or recounted')
        if self.cases[case_id].discrepancy_qty == 0:
            raise ValueError('discrepancy_qty == 0')


        self.cases[case_id].status = "ready_for_adjustment"

    def adjust_case(self, case_id: str, decision: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status != 'ready_for_adjustment':
            raise ValueError('not ready_for_adjustment')
        
        case = self.cases[case_id]
        case.status = "adjusted"
        case.decision = decision

    def escalate_case(self, case_id: str) -> None:
        if case_id not in self.cases:
            raise ValueError("Case not found")
        if self.cases[case_id].status  in self.done_statuses:
            raise ValueError('already done')
        if self.cases[case_id].status not in {'auditing','recounted','ready_for_adjustment'}:
            raise ValueError('can`t escalate')
        
        self.cases[case_id].status = "escalated"

    def list_cases(self) -> list[str]:
        rows = []
        for case in self.cases.values():
            rows.append(
                f"{case.case_id} | {case.sku} | {case.warehouse} | {case.expected_qty} | {case.counted_qty} | "
                f"{case.discrepancy_qty} | {case.discrepancy_value} | {case.status} | {case.auditor} | {case.recount_done} | {case.decision}"
            )
        return rows

    def summary(self) -> dict[str, float | int]:
        result = {
            "total_cases": 0,
            "total_loss_value": 0.0,
            "total_excess_value": 0.0,
        }
        for case in self.cases.values():
            result[case.status] = result.get(case.status, 0) + 1
            result["total_cases"] += 1
            if case.discrepancy_qty < 0:
                result["total_excess_value"] += case.discrepancy_value
            elif case.discrepancy_qty > 0:
                result["total_loss_value"] += case.discrepancy_value
        return result


class WarehouseAuditView:
    @staticmethod
    def render_cases(rows: list[str]) -> None:
        print("Warehouse audit cases:")
        for row in rows:
            print(row)

    @staticmethod
    def render_summary(summary: dict[str, float | int]) -> None:
        print("Summary:", summary)

    @staticmethod
    def render_success(message: str) -> None:
        print("SUCCESS:", message)

    @staticmethod
    def render_error(message: str) -> None:
        print("ERROR:", message)


class WarehouseAuditController:
    def __init__(self, model: WarehouseAuditModel, view: WarehouseAuditView):
        self.model = model
        self.view = view

    def create_case(self, case_id: str, sku: str, warehouse: str, expected_qty: int, counted_qty: int, unit_cost: float) -> None:
        try:
            self.model.add_case(case_id, sku, warehouse, expected_qty, counted_qty, unit_cost)
            self.view.render_success(f"Case {case_id} created")
        except ValueError as error:
            self.view.render_error(str(error))

    def assign_auditor(self, case_id: str, auditor: str) -> None:
        try:
            self.model.assign_auditor(case_id, auditor)
            self.view.render_success(f"Auditor assigned to {case_id}")
        except ValueError as error:
            self.view.render_error(str(error))

    def start_audit(self, case_id: str) -> None:
        try:
            self.model.start_audit(case_id)
            self.view.render_success(f"Case {case_id} moved to auditing")
        except ValueError as error:
            self.view.render_error(str(error))

    def recount_case(self, case_id: str, new_counted_qty: int) -> None:
        try:
            self.model.recount_case(case_id, new_counted_qty)
            self.view.render_success(f"Case {case_id} recounted")
        except ValueError as error:
            self.view.render_error(str(error))

    def mark_ready_for_adjustment(self, case_id: str) -> None:
        try:
            self.model.mark_ready_for_adjustment(case_id)
            self.view.render_success(f"Case {case_id} ready for adjustment")
        except ValueError as error:
            self.view.render_error(str(error))

    def adjust_case(self, case_id: str, decision: str) -> None:
        try:
            self.model.adjust_case(case_id, decision)
            self.view.render_success(f"Case {case_id} adjusted")
        except ValueError as error:
            self.view.render_error(str(error))

    def escalate_case(self, case_id: str) -> None:
        try:
            self.model.escalate_case(case_id)
            self.view.render_success(f"Case {case_id} escalated")
        except ValueError as error:
            self.view.render_error(str(error))

    def show_cases(self) -> None:
        self.view.render_cases(self.model.list_cases())
        self.view.render_summary(self.model.summary())


model = WarehouseAuditModel()
view = WarehouseAuditView()
controller = WarehouseAuditController(model, view)

for case_id, sku, warehouse, expected_qty, counted_qty, unit_cost in initial_cases:
    model.add_case(case_id, sku, warehouse, expected_qty, counted_qty, unit_cost)

for action in actions:
    if action[0] == "show":
        controller.show_cases()
    elif action[0] == "create":
        _, case_id, sku, warehouse, expected_qty, counted_qty, unit_cost = action
        controller.create_case(case_id, sku, warehouse, expected_qty, counted_qty, unit_cost)
    elif action[0] == "assign":
        _, case_id, auditor = action
        controller.assign_auditor(case_id, auditor)
    elif action[0] == "audit":
        _, case_id = action
        controller.start_audit(case_id)
    elif action[0] == "recount":
        _, case_id, new_counted_qty = action
        controller.recount_case(case_id, new_counted_qty)
    elif action[0] == "ready":
        _, case_id = action
        controller.mark_ready_for_adjustment(case_id)
    elif action[0] == "adjust":
        _, case_id, decision = action
        controller.adjust_case(case_id, decision)
    elif action[0] == "escalate":
        _, case_id = action
        controller.escalate_case(case_id)

print()
print("Финальное состояние:")
controller.show_cases()


Warehouse audit cases:
WA-100 | SKU-001 | MSK-A | 120 | 108 | -12 | 180.0 | new | None | False | None
WA-101 | SKU-002 | SPB-2 | 80 | 92 | 12 | 270.0 | new | None | False | None
Summary: {'total_cases': 2, 'total_loss_value': 270.0, 'total_excess_value': 180.0, 'new': 2}
ERROR: no auditor
SUCCESS: Auditor assigned to WA-100
SUCCESS: Case WA-100 moved to auditing
SUCCESS: Case WA-100 ready for adjustment
ERROR: not auditing
ERROR: not "auditing" or recounted
SUCCESS: Case WA-100 adjusted
SUCCESS: Case WA-102 created
SUCCESS: Auditor assigned to WA-102
SUCCESS: Case WA-102 moved to auditing
ERROR: discrepancy_qty == 0
SUCCESS: Case WA-102 escalated
Warehouse audit cases:
WA-100 | SKU-001 | MSK-A | 120 | 108 | -12 | 180.0 | adjusted | Olga | False | inventory_writeoff
WA-101 | SKU-002 | SPB-2 | 80 | 92 | 12 | 270.0 | new | None | False | None
WA-102 | SKU-003 | EKB-1 | 40 | 40 | 0 | 0.0 | escalated | Max | False | None
Summary: {'total_cases': 3, 'total_loss_value': 270.0, 'total_excess_v

In [ ]:
# Warehouse audit cases:
# WA-100 | SKU-001 | MSK-A | 120 | 108 | -12 | 180.0 | new | None | False | None
# WA-101 | SKU-002 | SPB-2 | 80 | 92 | 12 | 270.0 | new | None | False | None
# Summary: {'total_cases': 2, 'total_loss_value': 180.0, 'total_excess_value': 270.0, 'new': 2}
# ERROR: Auditor is required
# SUCCESS: Auditor assigned to WA-100
# SUCCESS: Case WA-100 moved to auditing
# SUCCESS: Case WA-100 ready for adjustment
# ERROR: Case is not auditing
# ERROR: Case is not ready for adjustment
# SUCCESS: Case WA-100 adjusted
# SUCCESS: Case WA-102 created
# SUCCESS: Auditor assigned to WA-102
# SUCCESS: Case WA-102 moved to auditing
# ERROR: No discrepancy to adjust
# SUCCESS: Case WA-102 escalated
# Warehouse audit cases:
# WA-100 | SKU-001 | MSK-A | 120 | 108 | -12 | 180.0 | adjusted | Olga | False | inventory_writeoff
# WA-101 | SKU-002 | SPB-2 | 80 | 92 | 12 | 270.0 | new | None | False | None
# WA-102 | SKU-003 | EKB-1 | 40 | 40 | 0 | 0.0 | escalated | Max | False | escalated
# Summary: {'total_cases': 3, 'total_loss_value': 180.0, 'total_excess_value': 270.0, 'adjusted': 1, 'new': 1, 'escalated': 1}
#
# Финальное состояние:
# Warehouse audit cases:
# WA-100 | SKU-001 | MSK-A | 120 | 108 | -12 | 180.0 | adjusted | Olga | False | inventory_writeoff
# WA-101 | SKU-002 | SPB-2 | 80 | 92 | 12 | 270.0 | new | None | False | None
# WA-102 | SKU-003 | EKB-1 | 40 | 40 | 0 | 0.0 | escalated | Max | False | escalated
# Summary: {'total_cases': 3, 'total_loss_value': 180.0, 'total_excess_value': 270.0, 'adjusted': 1, 'new': 1, 'escalated': 1}